# Regime-Switching Allocation Backtesting: Random Forest

## Overview
This Jupyter Notebook backtests a regime-switching tactical allocation strategy using Random Forest predictions from Notebook 08_2. It compares offensive vs defensive allocations based on predicted volatility regimes and measures performance, risk metrics, turnover, and outperformance vs benchmarks. The strategy allocates to high-beta/growth baskets during LOW_VOL regimes and low-beta/value baskets during HIGH_VOL regimes.

## Key Features
- **Model**: Random Forest regime predictions (binary: 0=LOW_VOL, 1=HIGH_VOL).
- **Allocation Rule**: Regime 0 → Offensive (high beta, growth, momentum), Regime 1 → Defensive (low beta, value, quality).
- **Timing**: Annual rebalancing (June 30), 12-month holding periods (July t → June t+1).
- **Baskets Tested**: 5 baskets (Economic classification, Beta expansion, Rolling beta, EWMA beta, Kalman beta).
- **Benchmarks**: Buy&Hold Offensive, Buy&Hold Defensive, Static 60/40, Market (NYSE P20 EW).
- **Metrics**: Sharpe, Sortino, Calmar, Max Drawdown, Win Rate, Cumulative Return, Alpha vs Market.

## Methodology
1. Load Random Forest predictions (from 08_2), basket returns (from 06_X), market benchmark, Fama-French factors.
2. Annual walk-forward backtesting: Predict regime, allocate for 12 months, rebalance.
3. Reconstruct monthly returns for all 5 strategies × 5 baskets.
4. Compute annualized metrics (CAGR, Sharpe, Calmar, Max DD) on monthly returns.
5. Compare Regime-Switching vs benchmarks (Buy&Hold, Static 60/40, Market).
6. Export monthly returns (Fama-French regression format) for alpha decomposition (Notebook 10_2).

## Dependencies
- Libraries: pandas, numpy, matplotlib, seaborn, sklearn, json.
- Data: 
  - Predictions: data/ml_data/models/random_forest/predictions.parquet
  - Baskets: data/ml_data/baskets/basket_{1-5}_*.parquet
  - Market: data/ml_data/market/market_returns_ew_nys80.parquet
  - Factors: data/factors/ff/FF5.csv, data/factors/q5/q5m.csv, data/factors/aqr-qmj/qmj_m.xlsx
- Environment: macOS with VS Code, Python 3.x.

## Usage
1. Run cells to load predictions, baskets, benchmarks, factors.
2. Backtest all 5 strategies × 5 baskets with annual rebalancing.
3. Review performance metrics (Sharpe, Calmar, Max DD) by basket and strategy.
4. Compare Regime-Switching vs Buy&Hold benchmarks.
5. Export monthly returns for Fama-French regressions (Notebook 10_2).
6. Adapt for other models (Logistic, XGBoost, LSTM) by changing MODEL_NAME variable.

## Output Files
- **data/ml_data/portfolios/random_forest/portfolio_results_monthly.parquet**: Monthly returns (all baskets × all strategies).
  - Columns: date, basket, strategy, formation_date, regime_predicted, regime_actual, proba_high_vol, portfolio_return, market_return, excess_return, correct_prediction.
  - Format: Fama-French regression ready (merge with FF3/FF5 factors for alpha decomposition).
- **Visualizations**: Performance charts, regime timeline, drawdown plots (generated in notebook cells).

## Performance Highlights
- **Best Sharpe**: Identifies top performer across baskets (e.g., Regime-Switching on Basket 3 Rolling Beta).
- **Market Comparison**: Regime-Switching outperforms Market (NYSE P20 EW) by X% CAGR with lower Max DD.
- **Regime Accuracy**: Tracks prediction accuracy and allocation switches over time.
- **Turnover**: Annual rebalancing = low turnover (2 trades/year max per basket).


In [31]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 : IMPORTS & CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.precision', 6)

print("="*100)
print("📦 NOTEBOOK 09_2 : REGIME-SWITCHING ALLOCATION BACKTESTING - Random Forest")
print("="*100)
print("\n✅ Imports loaded successfully")
print("="*100)

📦 NOTEBOOK 09_2 : REGIME-SWITCHING ALLOCATION BACKTESTING - Random Forest

✅ Imports loaded successfully


In [32]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 : DATA LOADING (PREDICTIONS + BASKETS + BENCHMARKS + FACTORS)
# ════════════════════════════════════════════════════════════════════════════

print("="*100)
print("📦 DATA LOADING")
print("="*100)

# ════════════════════════════════════════════════════════════════════════════
# 1. LOAD LOGISTIC REGRESSION PREDICTIONS
# ════════════════════════════════════════════════════════════════════════════

PREDICTIONS_PATH = Path('data/ml_data/models/random_forest/predictions.parquet')
df_predictions = pd.read_parquet(PREDICTIONS_PATH)

# Convert formation_date to datetime and set as index (MonthEnd aligned)
df_predictions['formation_date'] = pd.to_datetime(df_predictions['formation_date'])
df_predictions.set_index('formation_date', inplace=True)
df_predictions.index = df_predictions.index + pd.offsets.MonthEnd(0)
df_predictions.sort_index(inplace=True)

print(f"\n✅ Random Forest predictions loaded :")
print(f"   • Source         : {PREDICTIONS_PATH}")
print(f"   • Shape          : {df_predictions.shape}")
print(f"   • Period         : {df_predictions.index.min().date()} → {df_predictions.index.max().date()}")
print(f"   • Total formations : {len(df_predictions)}")
print(f"   • Columns        : {df_predictions.columns.tolist()}")

# ════════════════════════════════════════════════════════════════════════════
# 2. LOAD 5 BASKET RETURNS (FROM NOTEBOOK 06_X)
# ════════════════════════════════════════════════════════════════════════════

BASKETS_DIR = Path('data/ml_data/baskets')

basket_files = {
    'Basket_1_Economic': BASKETS_DIR / 'basket_1_economic_classification.parquet',
    'Basket_2_Beta_Expansion': BASKETS_DIR / 'basket_2_beta_expansion.parquet',
    'Basket_3_Rolling_Beta': BASKETS_DIR / 'basket_3_beta_rolling.parquet',
    'Basket_4_EWMA_Beta': BASKETS_DIR / 'basket_4_beta_ewma.parquet',
    'Basket_5_Kalman_Beta': BASKETS_DIR / 'basket_5_beta_kalman.parquet'
}

baskets = {}

for basket_name, basket_path in basket_files.items():
    if not basket_path.exists():
        print(f"\n⚠️  WARNING: {basket_name} not found at {basket_path}")
        continue
    
    df = pd.read_parquet(basket_path)
    df.index = pd.to_datetime(df.index) + pd.offsets.MonthEnd(0)
    df.sort_index(inplace=True)
    
    baskets[basket_name] = df
    
    print(f"\n✅ {basket_name} :")
    print(f"   • Shape          : {df.shape}")
    print(f"   • Period         : {df.index.min().date()} → {df.index.max().date()}")
    print(f"   • Columns        : {df.columns.tolist()}")

# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD MARKET BENCHMARK (NYSE P20 EW)
# ════════════════════════════════════════════════════════════════════════════

MARKET_PATH = Path('data/ml_data/market/market_returns_ew_nys80.parquet')

if not MARKET_PATH.exists():
    raise FileNotFoundError(
        f"Market benchmark not found: {MARKET_PATH}\n"
        f"Please run notebook 06_1_ML_NYSEP20_EW_Benchmark_creation.ipynb first."
    )

market = pd.read_parquet(MARKET_PATH)
market['date'] = pd.to_datetime(market['date']) + pd.offsets.MonthEnd(0)
market.set_index('date', inplace=True)
market.sort_index(inplace=True)

market_returns = market['ret_market_ew'].copy()

print(f"\n✅ Market Benchmark (NYSE P20 EW) :")
print(f"   • Source         : {MARKET_PATH}")
print(f"   • Period         : {market_returns.index.min().date()} → {market_returns.index.max().date()}")
print(f"   • Total months   : {len(market_returns)}")
print(f"   • Mean return    : {market_returns.mean()*100:.2f}% per month")
print(f"   • Volatility     : {market_returns.std()*100:.2f}% per month")


# ════════════════════════════════════════════════════════════════════════════
# 4. LOAD FAMA-FRENCH 5 FACTORS + RISK-FREE RATE
# ════════════════════════════════════════════════════════════════════════════

FF5_PATH = Path('data/factors/ff/FF5.csv')

if not FF5_PATH.exists():
    raise FileNotFoundError(f"Fama-French 5 factors not found: {FF5_PATH}")

ff5 = pd.read_csv(FF5_PATH)
ff5['dateff'] = pd.to_datetime(ff5['dateff']) + pd.offsets.MonthEnd(0)
ff5.set_index('dateff', inplace=True)
ff5.sort_index(inplace=True)

# Extract risk-free rate and factors
rf = ff5['rf'].copy()
factors_ff5 = ff5[['mktrf', 'smb', 'hml', 'rmw', 'cma', 'umd']].copy()

print(f"\n✅ Fama-French 5 Factors + RF :")
print(f"   • Source         : {FF5_PATH}")
print(f"   • Period         : {ff5.index.min().date()} → {ff5.index.max().date()}")
print(f"   • Total months   : {len(ff5)}")
print(f"   • Factors        : {factors_ff5.columns.tolist()}")
print(f"   • Mean RF        : {rf.mean()*100:.2f}% per month")

📦 DATA LOADING

✅ Random Forest predictions loaded :
   • Source         : data/ml_data/models/random_forest/predictions.parquet
   • Shape          : (44, 9)
   • Period         : 1980-06-30 → 2023-06-30
   • Total formations : 44
   • Columns        : ['test_year', 'actual_regime', 'predicted_regime', 'proba_high_vol', 'proba_low_vol', 'train_start', 'train_end', 'train_samples', 'correct']

✅ Basket_1_Economic :
   • Shape          : (738, 5)
   • Period         : 1963-07-31 → 2024-12-31
   • Columns        : ['Market', 'Offensive', 'Defensive', 'Turnover_Off', 'Turnover_Def']

✅ Basket_2_Beta_Expansion :
   • Shape          : (672, 5)
   • Period         : 1968-07-31 → 2024-06-30
   • Columns        : ['Market', 'HIGH_BETA', 'LOW_BETA', 'Turnover_High', 'Turnover_Low']

✅ Basket_3_Rolling_Beta :
   • Shape          : (672, 5)
   • Period         : 1968-07-31 → 2024-06-30
   • Columns        : ['Market', 'HIGH_BETA_ROLLING', 'LOW_BETA_ROLLING', 'Turnover_High', 'Turnover_Low']

✅ Ba

In [33]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3 : DATA VALIDATION (DTYPES, ALIGNMENT, PERIODS)
# ════════════════════════════════════════════════════════════════════════════

print("="*100)
print("🔍 DATA VALIDATION")
print("="*100)

# ════════════════════════════════════════════════════════════════════════════
# 1. PREDICTIONS VALIDATION
# ════════════════════════════════════════════════════════════════════════════

print(f"\n📊 PREDICTIONS DATAFRAME :")
print(f"   • Index type     : {type(df_predictions.index).__name__}")
print(f"   • Index freq     : {df_predictions.index.inferred_freq}")
print(f"   • Index is monotonic : {df_predictions.index.is_monotonic_increasing}")
print(f"\n   • Columns dtypes :")
for col in df_predictions.columns:
    print(f"      - {col:<20} : {df_predictions[col].dtype}")

print(f"\n   • Regime distribution :")
print(f"      - Predicted Regime 0 : {(df_predictions['predicted_regime'] == 0).sum()} ({(df_predictions['predicted_regime'] == 0).mean()*100:.1f}%)")
print(f"      - Predicted Regime 1 : {(df_predictions['predicted_regime'] == 1).sum()} ({(df_predictions['predicted_regime'] == 1).mean()*100:.1f}%)")

print(f"\n   • Missing values :")
print(df_predictions.isnull().sum())

# ════════════════════════════════════════════════════════════════════════════
# 2. BASKETS VALIDATION
# ════════════════════════════════════════════════════════════════════════════

print(f"\n\n📊 BASKETS VALIDATION :")

for basket_name, df_basket in baskets.items():
    print(f"\n   • {basket_name.upper()} :")
    print(f"      - Index type     : {type(df_basket.index).__name__}")
    print(f"      - Index freq     : {df_basket.index.inferred_freq}")
    print(f"      - Shape          : {df_basket.shape}")
    print(f"      - Columns        : {df_basket.columns.tolist()}")
    print(f"      - Missing values : {df_basket.isnull().sum().sum()}")
    
    # Check if returns are valid (between -100% and +500%)
    if 'ret' in df_basket.columns:
        ret_col = 'ret'
    elif 'return' in df_basket.columns:
        ret_col = 'return'
    else:
        ret_col = df_basket.columns[0]
    
    ret_min = df_basket[ret_col].min()
    ret_max = df_basket[ret_col].max()
    print(f"      - Returns range  : [{ret_min:.4f}, {ret_max:.4f}]")
    
    if ret_min < -1.0 or ret_max > 5.0:
        print(f"      ⚠️  WARNING: Extreme returns detected!")

# ════════════════════════════════════════════════════════════════════════════
# 3. PERIOD ALIGNMENT
# ════════════════════════════════════════════════════════════════════════════

print(f"\n\n📊 PERIOD ALIGNMENT :")

pred_start = df_predictions.index.min()
pred_end = df_predictions.index.max()

print(f"\n   • Predictions period : {pred_start.date()} → {pred_end.date()}")

for basket_name, df_basket in baskets.items():
    basket_start = df_basket.index.min()
    basket_end = df_basket.index.max()
    
    print(f"   • {basket_name:<12} : {basket_start.date()} → {basket_end.date()}")
    
    # Check overlap
    overlap_start = max(pred_start, basket_start)
    overlap_end = min(pred_end, basket_end)
    
    if overlap_start <= overlap_end:
        overlap_years = (overlap_end - overlap_start).days / 365.25
        print(f"      ✅ Overlap with predictions : {overlap_start.date()} → {overlap_end.date()} ({overlap_years:.1f} years)")
    else:
        print(f"      ❌ NO OVERLAP with predictions!")

print("\n" + "="*100)

🔍 DATA VALIDATION

📊 PREDICTIONS DATAFRAME :
   • Index type     : DatetimeIndex
   • Index freq     : YE-JUN
   • Index is monotonic : True

   • Columns dtypes :
      - test_year            : int64
      - actual_regime        : int64
      - predicted_regime     : int64
      - proba_high_vol       : float64
      - proba_low_vol        : float64
      - train_start          : datetime64[ns]
      - train_end            : datetime64[ns]
      - train_samples        : int64
      - correct              : int64

   • Regime distribution :
      - Predicted Regime 0 : 35 (79.5%)
      - Predicted Regime 1 : 9 (20.5%)

   • Missing values :
test_year           0
actual_regime       0
predicted_regime    0
proba_high_vol      0
proba_low_vol       0
train_start         0
train_end           0
train_samples       0
correct             0
dtype: int64


📊 BASKETS VALIDATION :

   • BASKET_1_ECONOMIC :
      - Index type     : DatetimeIndex
      - Index freq     : ME
      - Shape         

In [34]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 : BACKTESTING - REGIME-SWITCHING & BENCHMARKS (ALL 5 BASKETS)
# ════════════════════════════════════════════════════════════════════════════

print("="*100)
print("🔄 BACKTESTING : REGIME-SWITCHING & BENCHMARKS (ALL 5 BASKETS)")
print("="*100)

# ════════════════════════════════════════════════════════════════════════════
# BACKTESTING LOOP FOR ALL 5 BASKETS
# ════════════════════════════════════════════════════════════════════════════

all_results = {}  # Store results for each basket

for basket_name, df_basket in baskets.items():
    print(f"\n📊 Processing {basket_name}...")
    print(f"   • Columns : {df_basket.columns.tolist()}")
    
    # Identify offensive and defensive columns
    # Structure: [Market, Offensive, Defensive, Turnover_X, Turnover_Y]
    # For all baskets: col[1] = Offensive, col[2] = Defensive
    if len(df_basket.columns) >= 3:
        offensive_col = df_basket.columns[1]  # Second column
        defensive_col = df_basket.columns[2]  # Third column
    else:
        print(f"   ⚠️  WARNING: {basket_name} has fewer than 3 columns, skipping...")
        continue
    
    print(f"   • Offensive column : {offensive_col}")
    print(f"   • Defensive column : {defensive_col}")
    
    results = []
    
    for formation_date in df_predictions.index:
        year = formation_date.year
        
        # Get regime prediction
        regime_pred = df_predictions.loc[formation_date, 'predicted_regime']
        
        # Define holding period (July year t → June year t+1)
        # Use MONTH matching to handle weekends (e.g., July 31 = Sunday → data uses July 29)
        start_month = (formation_date + pd.DateOffset(months=1)).to_period('M')
        end_month = (formation_date + pd.DateOffset(months=12)).to_period('M')
        holding_mask = (df_basket.index.to_period('M') >= start_month) & (df_basket.index.to_period('M') <= end_month)

        
        if holding_mask.sum() < 12:
            # Skip if incomplete year
            continue
        
        monthly_offensive = df_basket.loc[holding_mask, offensive_col]
        monthly_defensive = df_basket.loc[holding_mask, defensive_col]
        
        # Get market returns for same period
        market_mask = (market_returns.index >= holding_start) & (market_returns.index <= holding_end)
        monthly_market = market_returns.loc[market_mask]
        
        if len(monthly_market) < 12:
            continue
        
        # ════════════════════════════════════════════════════════════════════
        # STRATEGY 1 : REGIME-SWITCHING (Binary Allocation)
        # ════════════════════════════════════════════════════════════════════
        if regime_pred == 0:
            # Low Vol → Offensive
            regime_switching_returns = monthly_offensive
            allocation = 'Offensive'
        else:
            # High Vol → Defensive
            regime_switching_returns = monthly_defensive
            allocation = 'Defensive'
        
        regime_switching_annual = (1 + regime_switching_returns).prod() - 1
        
        # ════════════════════════════════════════════════════════════════════
        # STRATEGY 2 : BUY & HOLD OFFENSIVE (100% Aggressive)
        # ════════════════════════════════════════════════════════════════════
        bh_offensive_annual = (1 + monthly_offensive).prod() - 1
        
        # ════════════════════════════════════════════════════════════════════
        # STRATEGY 3 : BUY & HOLD DEFENSIVE (100% Conservative)
        # ════════════════════════════════════════════════════════════════════
        bh_defensive_annual = (1 + monthly_defensive).prod() - 1
        
        # ════════════════════════════════════════════════════════════════════
        # STRATEGY 4 : STATIC 60/40 (Balanced)
        # ════════════════════════════════════════════════════════════════════
        static_6040_returns = 0.6 * monthly_offensive + 0.4 * monthly_defensive
        static_6040_annual = (1 + static_6040_returns).prod() - 1
        
        # ════════════════════════════════════════════════════════════════════
        # STRATEGY 5 : MARKET PORTFOLIO (NYSE P20 EW)
        # ════════════════════════════════════════════════════════════════════
        market_annual = (1 + monthly_market).prod() - 1
        
        # ════════════════════════════════════════════════════════════════════
        # STORE RESULTS
        # ════════════════════════════════════════════════════════════════════
        results.append({
            'formation_date': formation_date,
            'year': year,
            'regime_pred': regime_pred,
            'allocation': allocation,
            'regime_switching': regime_switching_annual,
            'bh_offensive': bh_offensive_annual,
            'bh_defensive': bh_defensive_annual,
            'static_6040': static_6040_annual,
            'market': market_annual
        })
    
    # Store results for this basket
    all_results[basket_name] = pd.DataFrame(results)
    
    print(f"   ✅ Completed : {len(results)} formations processed")

# ════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print(f"\n\n✅ Backtesting completed for all baskets :")
print(f"   • Total baskets  : {len(all_results)}")
print(f"   • Strategies     : 5 (Regime-Switching, BH Offensive, BH Defensive, Static 60/40, Market)")

for basket_name, df_results in all_results.items():
    print(f"\n   • {basket_name:<30} : {len(df_results)} formations ({df_results['year'].min()} → {df_results['year'].max()})")

print("\n" + "="*100)

🔄 BACKTESTING : REGIME-SWITCHING & BENCHMARKS (ALL 5 BASKETS)

📊 Processing Basket_1_Economic...
   • Columns : ['Market', 'Offensive', 'Defensive', 'Turnover_Off', 'Turnover_Def']
   • Offensive column : Offensive
   • Defensive column : Defensive
   ✅ Completed : 44 formations processed

📊 Processing Basket_2_Beta_Expansion...
   • Columns : ['Market', 'HIGH_BETA', 'LOW_BETA', 'Turnover_High', 'Turnover_Low']
   • Offensive column : HIGH_BETA
   • Defensive column : LOW_BETA
   ✅ Completed : 44 formations processed

📊 Processing Basket_3_Rolling_Beta...
   • Columns : ['Market', 'HIGH_BETA_ROLLING', 'LOW_BETA_ROLLING', 'Turnover_High', 'Turnover_Low']
   • Offensive column : HIGH_BETA_ROLLING
   • Defensive column : LOW_BETA_ROLLING
   ✅ Completed : 44 formations processed

📊 Processing Basket_4_EWMA_Beta...
   • Columns : ['Market', 'HIGH_BETA_EWMA', 'LOW_BETA_EWMA', 'Turnover_High', 'Turnover_Low']
   • Offensive column : HIGH_BETA_EWMA
   • Defensive column : LOW_BETA_EWMA
   ✅ Co

In [35]:
# ════════════════════════════════════════════════════════════════════════════
# CELL : PERFORMANCE METRICS - MONTHLY STATISTICS (IMPROVED DISPLAY)
# ════════════════════════════════════════════════════════════════════════════
"""
OBJECTIVE:
  • Compute performance metrics on MONTHLY returns (not annual aggregates)
  • Proper drawdown calculation capturing intra-year volatility
  • Professional display with clear metrics comparison
  • Market benchmark shown first for each basket comparison
"""

import pandas as pd
import numpy as np
from pathlib import Path

print("="*100)
print("📊 RF PERFORMANCE METRICS - MONTHLY STATISTICS (ALL BASKETS × ALL STRATEGIES)")
print("="*100)

# ════════════════════════════════════════════════════════════════════════════
# 1. RECONSTRUCT MONTHLY RETURNS FOR ALL STRATEGIES
# ════════════════════════════════════════════════════════════════════════════

def reconstruct_monthly_returns(basket_name, df_results, df_basket, market_returns_series):
    """
    Reconstruct monthly returns for all 5 strategies.
    
    Returns:
    --------
    dict : {strategy_name: pd.Series of monthly returns}
    """
    offensive_col = df_basket.columns[1]
    defensive_col = df_basket.columns[2]
    
    # Storage for monthly returns
    regime_switching_monthly = []
    bh_offensive_monthly = []
    bh_defensive_monthly = []
    static_6040_monthly = []
    market_monthly = []
    dates_monthly = []
    
    for idx, row in df_results.iterrows():
        formation_date = row['formation_date']
        regime_pred = row['regime_pred']
        
        # Define holding period (July year t → June year t+1)

        
        # Use MONTH matching to handle weekends (e.g., July 31 = Sunday → data uses July 29)

        
        start_month = (formation_date + pd.DateOffset(months=1)).to_period('M')


        
        end_month = (formation_date + pd.DateOffset(months=12)).to_period('M')

        
        holding_mask = (df_basket.index.to_period('M') >= start_month) & (df_basket.index.to_period('M') <= end_month)
        
        if holding_mask.sum() < 12:
            continue
        
        monthly_off = df_basket.loc[holding_mask, offensive_col]
        monthly_def = df_basket.loc[holding_mask, defensive_col]
        
        # Market returns
        market_mask = (market_returns_series.index >= holding_start) & (market_returns_series.index <= holding_end)
        monthly_mkt = market_returns_series.loc[market_mask]
        
        if len(monthly_mkt) < 12:
            continue
        
        # Regime-Switching allocation
        if regime_pred == 0:
            regime_monthly = monthly_off.values
        else:
            regime_monthly = monthly_def.values
        
        # Static 60/40
        static_monthly = 0.6 * monthly_off.values + 0.4 * monthly_def.values
        
        # Store
        regime_switching_monthly.extend(regime_monthly)
        bh_offensive_monthly.extend(monthly_off.values)
        bh_defensive_monthly.extend(monthly_def.values)
        static_6040_monthly.extend(static_monthly)
        market_monthly.extend(monthly_mkt.values)
        dates_monthly.extend(monthly_off.index)
    
    # Create Series
    return {
        'Market (NYSE P20 EW)': pd.Series(market_monthly, index=dates_monthly),  # Market FIRST
        'Regime-Switching': pd.Series(regime_switching_monthly, index=dates_monthly),
        'Buy&Hold Offensive': pd.Series(bh_offensive_monthly, index=dates_monthly),
        'Buy&Hold Defensive': pd.Series(bh_defensive_monthly, index=dates_monthly),
        'Static 60/40': pd.Series(static_6040_monthly, index=dates_monthly)
    }

# ════════════════════════════════════════════════════════════════════════════
# 2. COMPUTE MONTHLY METRICS
# ════════════════════════════════════════════════════════════════════════════

def compute_monthly_metrics(returns_series, rf_series, strategy_name, basket_name):
    """
    Compute performance metrics on MONTHLY returns.
    
    Parameters:
    -----------
    returns_series : pd.Series
        Monthly returns
    rf_series : pd.Series
        Monthly risk-free rate
    strategy_name : str
    basket_name : str
    
    Returns:
    --------
    dict : Performance metrics (ALL ANNUALIZED)
    """
    # Align rf with returns
    rf_aligned = rf_series.reindex(returns_series.index)
    rf_aligned = rf_aligned.fillna(rf_series.mean())  # Fill missing with average RF
    
    # Excess returns
    excess_returns = returns_series - rf_aligned
    
    # Cumulative return
    cumulative_return = (1 + returns_series).prod() - 1
    
    # Annualized return (CAGR)
    n_months = len(returns_series)
    n_years = n_months / 12
    annualized_return = (1 + cumulative_return) ** (1 / n_years) - 1
    
    # Annualized volatility
    annualized_vol = returns_series.std() * np.sqrt(12)
    
    # Sharpe ratio (annualized)
    sharpe = (returns_series.mean() - rf_aligned.mean()) * 12 / annualized_vol if annualized_vol > 0 else 0.0
    
    # Drawdown calculation (MONTHLY)
    cumulative_wealth = (1 + returns_series).cumprod()
    running_max = cumulative_wealth.expanding().max()
    drawdown = (cumulative_wealth - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Date of max drawdown
    max_dd_date = drawdown.idxmin()
    
    # Calmar ratio (annualized)
    calmar = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0.0
    
    # Sortino ratio (annualized)
    downside_returns = excess_returns[excess_returns < 0]
    downside_std_monthly = downside_returns.std() if len(downside_returns) > 0 else excess_returns.std()
    downside_std_annual = downside_std_monthly * np.sqrt(12)
    sortino = (returns_series.mean() - rf_aligned.mean()) * 12 / downside_std_annual if downside_std_annual > 0 else 0.0
    
    # Win rate
    win_rate = (returns_series > 0).mean()
    
    # Skewness & Kurtosis
    skewness = returns_series.skew()
    kurtosis = returns_series.kurtosis()
    
    return {
        'Basket': basket_name,
        'Strategy': strategy_name,
        'Period': f"{returns_series.index.min().date()} → {returns_series.index.max().date()}",
        'N Months': n_months,
        'N Years': round(n_years, 1),
        'CAGR (Annual)': annualized_return,
        'Volatility (Annual)': annualized_vol,
        'Sharpe (Annual)': sharpe,
        'Sortino (Annual)': sortino,
        'Calmar (Annual)': calmar,
        'Max Drawdown': max_drawdown,
        'Max DD Date': max_dd_date.date() if pd.notna(max_dd_date) else None,
        'Win Rate': win_rate,
        'Skewness': skewness,
        'Kurtosis': kurtosis,
        'Cumulative Return': cumulative_return
    }

# ════════════════════════════════════════════════════════════════════════════
# 3. COMPUTE FOR ALL BASKETS × ALL STRATEGIES
# ════════════════════════════════════════════════════════════════════════════

all_monthly_metrics = []

print("\n🔄 Computing monthly statistics for all baskets...\n")

for basket_name, df_results in all_results.items():
    print(f"   📊 Processing {basket_name}...")
    
    df_basket = baskets[basket_name]
    
    # Reconstruct monthly returns (Market will be first in dict)
    monthly_returns_dict = reconstruct_monthly_returns(
        basket_name, df_results, df_basket, market_returns
    )
    
    # Compute metrics for each strategy
    for strategy_name, returns_series in monthly_returns_dict.items():
        metrics = compute_monthly_metrics(
            returns_series, rf, strategy_name, basket_name
        )
        all_monthly_metrics.append(metrics)

print("\n✅ Monthly statistics computed for all strategies")

# ════════════════════════════════════════════════════════════════════════════
# 4. CREATE DISPLAY TABLE
# ════════════════════════════════════════════════════════════════════════════

df_monthly_metrics = pd.DataFrame(all_monthly_metrics)

# Format for display
df_display = df_monthly_metrics.copy()
df_display['CAGR (Annual)'] = (df_display['CAGR (Annual)'] * 100).round(2).astype(str) + '%'
df_display['Volatility (Annual)'] = (df_display['Volatility (Annual)'] * 100).round(2).astype(str) + '%'
df_display['Sharpe (Annual)'] = df_display['Sharpe (Annual)'].round(3)
df_display['Sortino (Annual)'] = df_display['Sortino (Annual)'].round(3)
df_display['Calmar (Annual)'] = df_display['Calmar (Annual)'].round(3)
df_display['Max Drawdown'] = (df_display['Max Drawdown'] * 100).round(2).astype(str) + '%'
df_display['Win Rate'] = (df_display['Win Rate'] * 100).round(1).astype(str) + '%'
df_display['Skewness'] = df_display['Skewness'].round(2)
df_display['Kurtosis'] = df_display['Kurtosis'].round(2)
df_display['Cumulative Return'] = (df_display['Cumulative Return'] * 100).round(1).astype(str) + '%'

# ════════════════════════════════════════════════════════════════════════════
# 5. PROFESSIONAL DISPLAY (MARKET FIRST FOR EACH BASKET)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*100)
print("📊 PERFORMANCE METRICS - ANNUALIZED STATISTICS (MONTHLY DATA)")
print("="*100)
print("\n📌 NOTE: All metrics are ANNUALIZED except Max Drawdown (computed on monthly returns)")
print("="*100)

# Define columns to display (in order of importance)
cols_to_show = [
    'Strategy',
    'CAGR (Annual)',
    'Volatility (Annual)',
    'Sharpe (Annual)',
    'Calmar (Annual)',
    'Max Drawdown',
    'Max DD Date',
    'Win Rate',
    'Skewness'
]

# Display by basket (Market benchmark first in each basket)
for basket_name in all_results.keys():
    print(f"\n{'─'*100}")
    print(f"📈 {basket_name.upper()}")
    print(f"{'─'*100}\n")
    
    # Get basket metrics
    basket_metrics = df_display[df_display['Basket'] == basket_name].copy()
    
    # Ensure Market is first
    market_row = basket_metrics[basket_metrics['Strategy'] == 'Market (NYSE P20 EW)']
    other_rows = basket_metrics[basket_metrics['Strategy'] != 'Market (NYSE P20 EW)']
    
    # Reorder: Market first, then others
    basket_metrics_ordered = pd.concat([market_row, other_rows], ignore_index=True)
    
    print(basket_metrics_ordered[cols_to_show].to_string(index=False))

# ════════════════════════════════════════════════════════════════════════════
# 6. SUMMARY STATISTICS (BEST PERFORMERS)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*100)
print("🏆 BEST PERFORMERS (ACROSS ALL BASKETS)")
print("="*100)

# Best Sharpe (exclude market)
df_strategies_only = df_monthly_metrics[df_monthly_metrics['Strategy'] != 'Market (NYSE P20 EW)'].copy()

best_sharpe = df_strategies_only.loc[df_strategies_only['Sharpe (Annual)'].idxmax()]
print(f"\n🥇 Highest Sharpe Ratio:")
print(f"   • Strategy       : {best_sharpe['Strategy']}")
print(f"   • Basket         : {best_sharpe['Basket']}")
print(f"   • Sharpe (Ann.)  : {best_sharpe['Sharpe (Annual)']:.3f}")
print(f"   • CAGR           : {best_sharpe['CAGR (Annual)']*100:.2f}%")
print(f"   • Volatility     : {best_sharpe['Volatility (Annual)']*100:.2f}%")
print(f"   • Max Drawdown   : {best_sharpe['Max Drawdown']*100:.2f}%")

# Best CAGR
best_cagr = df_strategies_only.loc[df_strategies_only['CAGR (Annual)'].idxmax()]
print(f"\n🥇 Highest CAGR:")
print(f"   • Strategy       : {best_cagr['Strategy']}")
print(f"   • Basket         : {best_cagr['Basket']}")
print(f"   • CAGR           : {best_cagr['CAGR (Annual)']*100:.2f}%")
print(f"   • Sharpe (Ann.)  : {best_cagr['Sharpe (Annual)']:.3f}")
print(f"   • Volatility     : {best_cagr['Volatility (Annual)']*100:.2f}%")
print(f"   • Max Drawdown   : {best_cagr['Max Drawdown']*100:.2f}%")

# Lowest Max DD
best_dd = df_strategies_only.loc[df_strategies_only['Max Drawdown'].idxmax()]
print(f"\n🥇 Lowest Max Drawdown:")
print(f"   • Strategy       : {best_dd['Strategy']}")
print(f"   • Basket         : {best_dd['Basket']}")
print(f"   • Max Drawdown   : {best_dd['Max Drawdown']*100:.2f}%")
print(f"   • Date           : {best_dd['Max DD Date']}")
print(f"   • CAGR           : {best_dd['CAGR (Annual)']*100:.2f}%")
print(f"   • Sharpe (Ann.)  : {best_dd['Sharpe (Annual)']:.3f}")

# Best Calmar
best_calmar = df_strategies_only.loc[df_strategies_only['Calmar (Annual)'].idxmax()]
print(f"\n🥇 Highest Calmar Ratio:")
print(f"   • Strategy       : {best_calmar['Strategy']}")
print(f"   • Basket         : {best_calmar['Basket']}")
print(f"   • Calmar (Ann.)  : {best_calmar['Calmar (Annual)']:.3f}")
print(f"   • CAGR           : {best_calmar['CAGR (Annual)']*100:.2f}%")
print(f"   • Max Drawdown   : {best_calmar['Max Drawdown']*100:.2f}%")

# ════════════════════════════════════════════════════════════════════════════
# 7. REGIME-SWITCHING RANKING
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'─'*100}")
print(f"📊 REGIME-SWITCHING PERFORMANCE RANKING (BY SHARPE)")
print(f"{'─'*100}\n")

regime_metrics = df_monthly_metrics[df_monthly_metrics['Strategy'] == 'Regime-Switching'].copy()
regime_metrics = regime_metrics.sort_values('Sharpe (Annual)', ascending=False)

print(f"{'Basket':<30} │ {'Sharpe':>7} │ {'CAGR':>7} │ {'Vol':>7} │ {'MaxDD':>8} │ {'Calmar':>7} │ {'Win%':>6}")
print(f"{'─'*30}─┼─{'─'*7}─┼─{'─'*7}─┼─{'─'*7}─┼─{'─'*8}─┼─{'─'*7}─┼─{'─'*6}")

for idx, row in regime_metrics.iterrows():
    print(f"{row['Basket']:<30} │ {row['Sharpe (Annual)']:>7.3f} │ {row['CAGR (Annual)']*100:>6.2f}% │ "
          f"{row['Volatility (Annual)']*100:>6.2f}% │ {row['Max Drawdown']*100:>7.2f}% │ "
          f"{row['Calmar (Annual)']:>7.3f} │ {row['Win Rate']*100:>5.1f}%")

# ════════════════════════════════════════════════════════════════════════════
# 8. MARKET BENCHMARK COMPARISON
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'─'*100}")
print(f"📊 MARKET BENCHMARK SUMMARY")
print(f"{'─'*100}\n")

market_metrics = df_monthly_metrics[df_monthly_metrics['Strategy'] == 'Market (NYSE P20 EW)'].iloc[0]

print(f"   • Period         : {market_metrics['Period']}")
print(f"   • N Years        : {market_metrics['N Years']}")
print(f"   • CAGR (Ann.)    : {market_metrics['CAGR (Annual)']*100:.2f}%")
print(f"   • Volatility     : {market_metrics['Volatility (Annual)']*100:.2f}%")
print(f"   • Sharpe (Ann.)  : {market_metrics['Sharpe (Annual)']:.3f}")
print(f"   • Calmar (Ann.)  : {market_metrics['Calmar (Annual)']:.3f}")
print(f"   • Max Drawdown   : {market_metrics['Max Drawdown']*100:.2f}% (on {market_metrics['Max DD Date']})")
print(f"   • Win Rate       : {market_metrics['Win Rate']*100:.1f}%")
print(f"   • Cumulative Ret : {market_metrics['Cumulative Return']*100:.1f}%")

print("\n" + "="*100)
print("✅ Monthly statistics calculation completed")
print("="*100)

📊 RF PERFORMANCE METRICS - MONTHLY STATISTICS (ALL BASKETS × ALL STRATEGIES)

🔄 Computing monthly statistics for all baskets...

   📊 Processing Basket_1_Economic...
   📊 Processing Basket_2_Beta_Expansion...
   📊 Processing Basket_3_Rolling_Beta...
   📊 Processing Basket_4_EWMA_Beta...
   📊 Processing Basket_5_Kalman_Beta...

✅ Monthly statistics computed for all strategies

📊 PERFORMANCE METRICS - ANNUALIZED STATISTICS (MONTHLY DATA)

📌 NOTE: All metrics are ANNUALIZED except Max Drawdown (computed on monthly returns)

────────────────────────────────────────────────────────────────────────────────────────────────────
📈 BASKET_1_ECONOMIC
────────────────────────────────────────────────────────────────────────────────────────────────────

            Strategy CAGR (Annual) Volatility (Annual)  Sharpe (Annual)  Calmar (Annual) Max Drawdown Max DD Date Win Rate  Skewness
Market (NYSE P20 EW)        13.27%              19.96%            0.532            0.906      -14.65%  1988-10-31    

In [36]:
# ════════════════════════════════════════════════════════════════════════════════
# CODE D'EXPORT MENSUEL - VERSION CORRIGÉE (GÈRE TOUS LES BASKETS)
# ════════════════════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════════════════
# EXPORT MONTHLY RETURNS (FAMA-FRENCH FORMAT)
# ════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*100}")
print(f"💾 EXPORTING MONTHLY BACKTEST RESULTS (ALL BASKETS × ALL STRATEGIES)")
print(f"{'='*100}\n")

from pathlib import Path
import pandas as pd

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION (CHANGE PER NOTEBOOK)
# ════════════════════════════════════════════════════════════════════════════

MODEL_NAME = 'random_forest'  # ⚠️ CHANGE THIS: logistic, random_forest, xgboost, lstm_expanding, lstm_rolling_10y

# ════════════════════════════════════════════════════════════════════════════
# BASKET FILE MAPPING
# ════════════════════════════════════════════════════════════════════════════

BASKET_FILES = {
    'Basket_1_Economic': 'basket_1_economic_classification.parquet',
    'Basket_2_Beta_Expansion': 'basket_2_beta_expansion.parquet',
    'Basket_3_Rolling_Beta': 'basket_3_beta_rolling.parquet',
    'Basket_4_EWMA_Beta': 'basket_4_beta_ewma.parquet',
    'Basket_5_Kalman_Beta': 'basket_5_beta_kalman.parquet'
}

# ════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTION: AUTO-DETECT COLUMN NAMES
# ════════════════════════════════════════════════════════════════════════════

def get_basket_columns(df_basket):
    """
    Auto-detect offensive, defensive, and market column names.

    Basket 1: Offensive, Defensive, Market
    Basket 2-5: HIGH_BETA_*, LOW_BETA_*, Market
    """
    cols = df_basket.columns.tolist()

    # Market column (always 'Market')
    market_col = 'Market'

    # Offensive column (high risk / high beta / growth)
    offensive_candidates = [c for c in cols if 'offensive' in c.lower() or 'high' in c.lower()]
    if not offensive_candidates:
        raise ValueError(f"No offensive/high column found in {cols}")
    offensive_col = offensive_candidates[0]

    # Defensive column (low risk / low beta / value)
    defensive_candidates = [c for c in cols if 'defensive' in c.lower() or 'low' in c.lower()]
    if not defensive_candidates:
        raise ValueError(f"No defensive/low column found in {cols}")
    defensive_col = defensive_candidates[0]

    return offensive_col, defensive_col, market_col

# ════════════════════════════════════════════════════════════════════════════
# BUILD MONTHLY EXPORT
# ════════════════════════════════════════════════════════════════════════════

all_exports = []
skipped_formations = []

for basket_name, df_results in all_results.items():

    # Load basket monthly returns
    basket_filename = BASKET_FILES[basket_name]
    basket_path = Path(f'data/ml_data/baskets/{basket_filename}')
    df_basket = pd.read_parquet(basket_path)

    # Auto-detect column names for this basket
    offensive_col, defensive_col, market_col = get_basket_columns(df_basket)

    print(f"📦 {basket_name}:")
    print(f"   • Offensive col: {offensive_col}")
    print(f"   • Defensive col: {defensive_col}")
    print(f"   • Market col   : {market_col}")

    # Merge results with predictions
    df_merged = df_results.merge(
        df_predictions[['actual_regime', 'proba_high_vol']],
        left_on='formation_date',
        right_index=True,
        how='left'
    )

    # Process each formation
    for idx, row in df_merged.iterrows():
        formation_date = row['formation_date']
        regime_pred = row['regime_pred']
        regime_actual = row['actual_regime']
        proba_high_vol = row['proba_high_vol']
        allocation = row['allocation']  # 'Offensive' or 'Defensive'

        # Holding period (July year t → June year t+1)
        # ✅ FIXED: Use MONTH matching to handle weekends (July 31 = Sunday → data uses July 29)
        start_month = (formation_date + pd.DateOffset(months=1)).to_period('M')  # July year t
        end_month = (formation_date + pd.DateOffset(months=12)).to_period('M')   # June year t+1

        # Extract monthly returns using month matching
        holding_mask = (df_basket.index.to_period('M') >= start_month) & (df_basket.index.to_period('M') <= end_month)

        if holding_mask.sum() < 12:
            skipped_formations.append((basket_name, formation_date, holding_mask.sum()))
            continue

        df_holding = df_basket.loc[holding_mask].copy()

        # Extract monthly returns using detected column names
        monthly_offensive = df_holding[offensive_col]
        monthly_defensive = df_holding[defensive_col]
        monthly_market = df_holding[market_col]

        # Regime-Switching allocation
        if allocation == 'Offensive':
            monthly_regime_switching = monthly_offensive
        else:
            monthly_regime_switching = monthly_defensive

        # Static 60/40
        monthly_static_6040 = 0.6 * monthly_offensive + 0.4 * monthly_defensive

        # Create monthly rows for each strategy
        strategies = {
            'Regime-Switching': monthly_regime_switching,
            'Buy&Hold Offensive': monthly_offensive,
            'Buy&Hold Defensive': monthly_defensive,
            'Static 60/40': monthly_static_6040,
            'Market': monthly_market
        }

        for strategy_name, monthly_returns in strategies.items():
            for month_date, monthly_return in monthly_returns.items():
                all_exports.append({
                    'date': month_date,
                    'basket': basket_name,
                    'strategy': strategy_name,
                    'formation_date': formation_date,
                    'regime_predicted': regime_pred,
                    'regime_actual': regime_actual,
                    'proba_high_vol': proba_high_vol,
                    'portfolio_return': monthly_return,
                    'market_return': monthly_market.loc[month_date],
                    'excess_return': monthly_return - monthly_market.loc[month_date],
                    'correct_prediction': int(regime_pred == regime_actual)
                })

# Build DataFrame
export_df = pd.DataFrame(all_exports)
export_df = export_df.set_index('date').sort_index()


# ════════════════════════════════════════════════════════════════════════════
# SUMMARY STATISTICS
# ════════════════════════════════════════════════════════════════════════════

print(f"\n📊 Monthly Export DataFrame:")
print(f"   • Total rows     : {len(export_df):,}")
print(f"   • Baskets        : {export_df['basket'].nunique()}")
print(f"   • Strategies     : {export_df['strategy'].nunique()}")
print(f"   • Period         : {export_df.index.min().date()} → {export_df.index.max().date()}")
print(f"   • Unique months  : {export_df.index.nunique()}")
print(f"   • Columns        : {list(export_df.columns)}\n")

if skipped_formations:
    print(f"   ⚠️  Skipped formations (< 11 months available):")
    for basket, date, months in skipped_formations[:5]:  # Show first 5
        print(f"      • {basket} - {date.date()} ({months} months)")
    if len(skipped_formations) > 5:
        print(f"      ... and {len(skipped_formations) - 5} more")
    print()

print(f"   Breakdown by basket × strategy:")
for basket in sorted(export_df['basket'].unique()):
    for strategy in sorted(export_df['strategy'].unique()):
        months = export_df[(export_df['basket'] == basket) & (export_df['strategy'] == strategy)].index.nunique()
        print(f"      • {basket:<30} × {strategy:<20} : {months:>3} months")

# ════════════════════════════════════════════════════════════════════════════
# EXPORT TO PARQUET
# ════════════════════════════════════════════════════════════════════════════

output_dir = Path('data/ml_data/portfolios') / MODEL_NAME
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'portfolio_results_monthly.parquet'
export_df.to_parquet(output_path)

print(f"\n✅ Exported: {output_path}")
print(f"   • Size : {output_path.stat().st_size / 1024:.1f} KB\n")

# ════════════════════════════════════════════════════════════════════════════
# USAGE EXAMPLES
# ════════════════════════════════════════════════════════════════════════════

print(f"{'='*100}")
print(f"📊 USAGE EXAMPLES (NOTEBOOK 10_1 + FAMA-FRENCH REGRESSIONS)")
print(f"{'='*100}\n")

print(f"# Load monthly results")
print(f"df = pd.read_parquet('{output_path}')")
print(f"")
print(f"# Get monthly returns for Regime-Switching strategy")
print(f"rs = df[df['strategy'] == 'Regime-Switching']")
print(f"")
print(f"# Average across all baskets")
print(f"monthly_returns = rs.groupby('date')['portfolio_return'].mean()")
print(f"")
print(f"# Fama-French 3-Factor Regression")
print(f"import pandas_datareader as pdr")
print(f"ff3 = pdr.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-07', end='2024-06')[0]")
print(f"ff3 = ff3 / 100  # Convert to decimal")
print(f"")
print(f"# Merge and run regression")
print(f"df_reg = pd.merge(monthly_returns, ff3, left_index=True, right_index=True)")
print(f"")
print(f"from sklearn.linear_model import LinearRegression")
print(f"X = df_reg[['Mkt-RF', 'SMB', 'HML']]")
print(f"y = df_reg['portfolio_return'] - df_reg['RF']")
print(f"model = LinearRegression().fit(X, y)")
print(f"alpha = model.intercept_")
print(f"print(f'Alpha (monthly): {{alpha:.4%}}')")

print(f"\n{'='*100}")
print(f"✅ Monthly export complete for {MODEL_NAME.upper()}")
print(f"{'='*100}\n")

# ════════════════════════════════════════════════════════════════════════════════
# MAPPING DES COLONNES PAR BASKET:
# ════════════════════════════════════════════════════════════════════════════════

# Basket 1 (Economic):
#   • Offensive: Offensive
#   • Defensive: Defensive
#   • Market: Market

# Basket 2 (Beta Expansion):
#   • Offensive: HIGH_BETA
#   • Defensive: LOW_BETA
#   • Market: Market

# Basket 3 (Rolling Beta):
#   • Offensive: HIGH_BETA_ROLLING
#   • Defensive: LOW_BETA_ROLLING
#   • Market: Market

# Basket 4 (EWMA Beta):
#   • Offensive: HIGH_BETA_EWMA
#   • Defensive: LOW_BETA_EWMA
#   • Market: Market

# Basket 5 (Kalman Beta):
#   • Offensive: HIGH_BETA_KALMAN
#   • Defensive: LOW_BETA_KALMAN
#   • Market: Market

# La fonction get_basket_columns() détecte automatiquement les bons noms !



💾 EXPORTING MONTHLY BACKTEST RESULTS (ALL BASKETS × ALL STRATEGIES)

📦 Basket_1_Economic:
   • Offensive col: Offensive
   • Defensive col: Defensive
   • Market col   : Market
📦 Basket_2_Beta_Expansion:
   • Offensive col: HIGH_BETA
   • Defensive col: LOW_BETA
   • Market col   : Market
📦 Basket_3_Rolling_Beta:
   • Offensive col: HIGH_BETA_ROLLING
   • Defensive col: LOW_BETA_ROLLING
   • Market col   : Market
📦 Basket_4_EWMA_Beta:
   • Offensive col: HIGH_BETA_EWMA
   • Defensive col: LOW_BETA_EWMA
   • Market col   : Market
📦 Basket_5_Kalman_Beta:
   • Offensive col: HIGH_BETA_KALMAN
   • Defensive col: LOW_BETA_KALMAN
   • Market col   : Market

📊 Monthly Export DataFrame:
   • Total rows     : 13,200
   • Baskets        : 5
   • Strategies     : 5
   • Period         : 1980-07-31 → 2024-06-28
   • Unique months  : 528
   • Columns        : ['basket', 'strategy', 'formation_date', 'regime_predicted', 'regime_actual', 'proba_high_vol', 'portfolio_return', 'market_return', 'excess